In [2]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Change working directory to project root so relative paths work
os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())

from backtesting.backtest_simulator import VectorizedBacktester

print(f"[SUCCESS] Working directory: {os.getcwd()}")
print("[SUCCESS] VectorizedBacktester imported.")

[SUCCESS] Working directory: /app
[SUCCESS] VectorizedBacktester imported.


In [ ]:
# Survival-constrained Monte Carlo optimization
# Max drawdown > -20% rejects the config entirely, even if RoMD is high
MAX_DRAWDOWN_THRESHOLD = -20.0  # percent
NUM_TESTS = 30  # each run takes a few minutes — start small

results = []

print(f"Running survival-constrained Monte Carlo ({NUM_TESTS} iterations)...\n")

for i in range(NUM_TESTS):
    # Parameter ranges — leverage stays reasonable for retail Reg-T
    test_z = round(random.uniform(1.8, 2.5), 2)
    test_ai = round(random.uniform(0.55, 0.68), 2)
    test_pt = round(random.uniform(1.0, 2.0), 2)
    test_sl = round(random.uniform(1.0, 2.0), 2)
    test_lev = round(random.uniform(1.5, 2.0), 2)  # Reg-T cap for retail Alpaca

    # Instantiate a fresh backtester with these parameters
    bt = VectorizedBacktester()
    bt.Z_THRESH = test_z
    bt.AI_THRESH = test_ai
    bt.PT_SKEW = test_pt
    bt.SL_SKEW = test_sl
    bt.LEVERAGE = test_lev

    # Run silently — we don't need the tearsheet for each iteration
    try:
        total_return_pct, max_dd_pct, num_trades = bt.run_simulation_headless()
    except Exception as e:
        print(f"[{i+1}/{NUM_TESTS}] FAILED: {e}")
        continue

    # survival constraint: reject configs that blow the account
    if max_dd_pct < MAX_DRAWDOWN_THRESHOLD:
        print(f"[{i+1}/{NUM_TESTS}] REJECTED: DD={max_dd_pct:.2f}% exceeds {MAX_DRAWDOWN_THRESHOLD}%")
        continue

    # Compute RoMD only for survivors
    romd = abs(total_return_pct / max_dd_pct) if max_dd_pct < 0 else 0.0

    results.append({
        "Z": test_z, "AI": test_ai, "PT": test_pt, "SL": test_sl, "Lev": test_lev,
        "Return%": round(total_return_pct, 2),
        "MaxDD%": round(max_dd_pct, 2),
        "Trades": num_trades,
        "RoMD": round(romd, 2)
    })
    print(f"[{i+1}/{NUM_TESTS}] Return={total_return_pct:.2f}% DD={max_dd_pct:.2f}% RoMD={romd:.2f}")

results_df = pd.DataFrame(results).sort_values(by="RoMD", ascending=False)
print(f"\n=== TOP 10 SURVIVABLE CONFIGURATIONS ===")
print(f"({len(results)} survived the DD < {MAX_DRAWDOWN_THRESHOLD}% constraint)\n")
results_df.head(10)

Running survival-constrained Monte Carlo (30 iterations)...

[SUCCESS] Loaded 41 historical baskets from structural_lifecycle_5yr.json.
[SUCCESS] Loaded XGBoost Meta-Labeler.
[SYSTEM] Loading historical matrix from the_execution_node/data/backtest_5m_5yr.parquet...
[SUCCESS] Historical matrix: 501636 rows x 26 cols
[1/30] Return=16.80% DD=-1.20% RoMD=14.03
[SUCCESS] Loaded 41 historical baskets from structural_lifecycle_5yr.json.
[SUCCESS] Loaded XGBoost Meta-Labeler.
[SYSTEM] Loading historical matrix from the_execution_node/data/backtest_5m_5yr.parquet...
[SUCCESS] Historical matrix: 501636 rows x 26 cols
[2/30] Return=20.22% DD=-1.36% RoMD=14.82
[SUCCESS] Loaded 41 historical baskets from structural_lifecycle_5yr.json.
[SUCCESS] Loaded XGBoost Meta-Labeler.
[SYSTEM] Loading historical matrix from the_execution_node/data/backtest_5m_5yr.parquet...
[SUCCESS] Historical matrix: 501636 rows x 26 cols
[3/30] Return=19.58% DD=-1.51% RoMD=13.01
[SUCCESS] Loaded 41 historical baskets from s

,Z,AI,PT,SL,Lev,Return%,MaxDD%,Trades,RoMD
5,2.39,0.56,1.90,1.75,2.00,31.58,-1.46,3534,21.64
11,2.22,0.56,1.14,1.09,1.97,29.91,-1.39,3870,21.52
15,2.17,0.59,1.67,1.95,1.82,24.93,-1.22,3738,20.36
12,2.22,0.57,1.57,1.31,1.60,22.90,-1.13,3784,20.21
25,2.20,0.57,1.42,1.53,1.62,22.92,-1.16,3818,19.75
7,2.49,0.59,1.72,1.39,1.79,25.74,-1.30,3127,19.75
28,2.28,0.62,1.94,1.36,1.88,25.64,-1.43,3298,17.94
4,2.30,0.64,1.81,1.27,1.94,26.19,-1.47,3086,17.80
24,2.10,0.59,1.80,1.25,1.54,19.74,-1.13,3866,17.42
23,2.33,0.63,1.91,1.48,1.88,25.85,-1.59,3112,16.29
